# Unit 08 - Power and MDE (Demo)

**Atoms served:** `U08-A1`, `U08-A2`, `U08-A3`, `U08-A4` (**MDE from business threshold - no video**; Kohavi ch. 17 and this notebook carry it)

**Estimated runtime:** ~20 seconds

**After this notebook you can:** read a power curve, compute sample size from `alpha`, `beta`, and effect size, and derive `MDE` from a ship threshold instead of a calculator default.

## Without code

1. Power curve: as true lift rises, power rises toward 1.
2. Sample size table: doubling `MDE` quarters required `n` (inverse square relationship).
3. Business `MDE`: if finance ships at +0.5% conversion, that number - not 0.1% - goes into the formula.

**Note:** `V28` covers `alpha`, `beta`, and `n = f(alpha, beta, effect)` but never names `MDE` tied to a business decision. This notebook fills that gap.

## 1. The question

Finance will ship a checkout change only if conversion rises by **at least 0.5 percentage points** (0.005 in proportion units). How many users do you need - and what happens if you peek at power curves without that threshold?

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Baseline conversion `p0 = 0.12`. We simulate Bernoulli outcomes under varying true lifts for power curves.

In [ ]:
p0 = 0.12
alpha = 0.05
beta = 0.20  # power = 1 - beta = 0.80
z_alpha = stats.norm.ppf(1 - alpha/2)
z_beta = stats.norm.ppf(1 - beta)
print('Baseline p0:', p0, 'alpha:', alpha, 'target power:', 1-beta)

## 4. The naive move

Use the online calculator default "minimum detectable effect" of 1% because it is pre-filled.

In [ ]:
default_mde = 0.01  # 1% absolute - calculator default
p1d = p0 + default_mde
n_default = ((z_alpha + z_beta) ** 2 * (p0 * (1 - p0) + p1d * (1 - p1d))) / default_mde ** 2
print('Required n per arm with 1% MDE:', int(np.ceil(n_default)))

The default `MDE` may detect lifts finance would ignore - wasting traffic on trivial wins.

## 5. What actually happens

**Power curves (`U08-A1`, `U08-A3`).** Power is the chance you detect a real effect.

In [ ]:
def power_two_proportion(p0, mde, n_per_arm, alpha=0.05):
    p1 = p0 + mde
    se0 = np.sqrt(p0*(1-p0)/n_per_arm + p0*(1-p0)/n_per_arm)
    crit = mde  # approximate for equal variance
    se1 = np.sqrt(p1*(1-p1)/n_per_arm + p0*(1-p0)/n_per_arm)
    z = (mde - z_alpha * se0) / se1
    return stats.norm.cdf(z)

n_arm = 5000
effect_grid = np.linspace(0.0, 0.02, 30)
powers = [power_two_proportion(p0, d, n_arm) for d in effect_grid]
fig, ax = plt.subplots()
ax.plot(effect_grid, powers)
ax.axvline(0.005, linestyle='--', label='business MDE 0.5pp')
ax.set_xlabel('true absolute lift')
ax.set_ylabel('power')
ax.set_title('Power rises with effect size at fixed n')
ax.legend()
plt.show()

Below the business threshold, power is low - you would correctly fail to ship noise. Above it, power climbs.

**`MDE` from business threshold (`U08-A4`) - not on video.** Start with the smallest lift you would act on.

In [ ]:
# 0.005 is the smallest lift finance would act on
business_mde = 0.005
p1 = p0 + business_mde
n_business = ((z_alpha + z_beta) ** 2 * (p0 * (1 - p0) + p1 * (1 - p1))) / business_mde ** 2
print('Business MDE (absolute):', business_mde)
print('Required n per arm:', int(np.ceil(n_business)))
print('Using 1% default would need only', int(np.ceil(n_default)), 'per arm - but detects effects you would not ship.')

Sample size scales like `1 / MDE^2`. Halving the `MDE` you care about quadruples `n` (`U08-A2`: more units shrink `SE`).

**Duration preview (`U08-A5`).** Weekly seasonality and novelty set floors; opportunity cost sets the ceiling - covered in Kohavi ch. 15, not on video.

In [ ]:
users_per_week = 8000
weeks_needed = int(np.ceil(n_business * 2 / users_per_week))  # two arms
print('At', users_per_week, 'assignments/week, run at least', weeks_needed, 'weeks for the business MDE.')
print('Shorter runs risk seasonality; longer runs pay opportunity cost (V29).')

## 6. What you do about it

1. Agree the **ship threshold** with the decision owner (`U08-A4`).
2. Plug that **`MDE`**, plus `alpha` and `beta`, into the sample-size formula (`U08-A3`, `V28`).
3. Translate `n` into **weeks** with traffic and seasonality in mind (`U08-A5`).

**When this matters less:** Exploratory tests with cheap reversals can use looser thresholds - but someone must still write them down.

---

**Takeaway:** Power is a design choice. `MDE` is a business conversation first and a formula second. `V28` gives the statistics; this notebook gives the threshold the statistics serve.

**Back to the unit:** [V1](../V1/units/unit-08-power-duration-sample-size/README.md) · [V2](../V2/units/unit-08-power-duration-sample-size/README.md)